# Sprint 4 — Sesión Práctica — Versión estudiante
## Funnel A/B y decisión de negocio con SQL

**Modalidad:** contexto guiado → breakout rooms → Coding Together  
**Duración:** 70 minutos  
**Motor:** DuckDB en Google Colab  
**Datos:** archivos `.parquet` con prefijo `sprint04_`

> **Hilo conductor:** comprender el experimento, explorar su esquema, construir el funnel y emitir una recomendación prudente.


## Cómo usar esta versión

- Lee el contexto antes de explorar las tablas.
- Durante el breakout, completa el inventario de las cuatro tablas.
- Las consultas centrales usan el patrón de plataforma: una CTE por etapa y `LEFT JOIN`.
- Registra tanto el resultado numérico como su significado de negocio.
- No consultes una solución final antes de documentar tu procedimiento.


## Contexto de negocio

Durante marzo de 2024 se ejecutó un experimento:

- **A:** checkout actual.
- **B:** checkout con menos pasos y mensajes más claros.
- **Etapa objetivo:** `begin_checkout → purchase`.

La dirección pregunta:

> **¿La variante B mejora el resultado del proceso y existe evidencia suficiente para desplegarla?**

### Hipótesis inicial

Creo que la variante ______ será mejor porque __________________________.

### Métrica que debería decidir el resultado

______________________________________________________________________


## Objetivos de aprendizaje

1. Explicar la granularidad y relación de las cuatro tablas.
2. Formular preguntas de exploración antes de construir métricas.
3. Construir un funnel A/B con una CTE por etapa y `LEFT JOIN`.
4. Calcular conversión entre etapas y conversión total.
5. Evaluar la etapa objetivo del experimento.
6. Comparar resultados por país sin confundir volumen con desempeño.
7. Redactar una recomendación con decisión, evidencia, limitación y siguiente acción.


## Reglas y entregable

1. El contexto y el esquema se revisan antes del breakout.
2. Todos los grupos exploran las cuatro tablas.
3. La exploración identifica granularidad, llaves y categorías; no se centra en limpieza.
4. El funnel se construye después en plenaria.
5. Cada resultado debe responder una pregunta de negocio.
6. Mayor revenue bruto no implica automáticamente mejor variante.

**Entregable**

- inventario de tablas;
- funnel A/B;
- tasas de conversión;
- evaluación de la etapa objetivo;
- lectura por país;
- recomendación ejecutiva.


## Preguntas guiadas

1. ¿Qué etapa intenta mejorar el experimento?
2. ¿Qué tabla contiene el significado de A y B?
3. ¿Qué tabla representa una sesión?
4. ¿Cuál es el denominador de `checkout → purchase`?
5. ¿Mayor revenue implica mejor conversión?

### Mis respuestas iniciales

1.  
2.  
3.  
4.  
5.  

## Estructura y timeboxing

| Tiempo | Bloque | Modalidad | Resultado |
|---:|---|---|---|
| 0–8 | Apertura | Plenaria | contexto e hipótesis |
| 8–15 | Bloque 0 | Plenaria | esquema y granularidad |
| 15–27 | Bloque 1 | Breakout rooms | exploración de cuatro tablas |
| 27–45 | Bloque 2 | Coding Together | funnel A/B |
| 45–58 | Bloque 3 | Coding Together | lectura del experimento |
| 58–64 | Bloque 4 | Coding Together | segmentación y recomendación |
| 64–70 | Cierre | Plenaria | validación |


<div style="text-align: center">
    <img src="https://raw.githubusercontent.com/ljpiere/tpdata_python/main/images/w1s1_2.png" width="400">
</div>


---
# Bloque 0 · Esquema antes de explorar

```text
users                                           funnel_sessions
────────────────────────                        ───────────────────────────
user_id            PK  ──────────────────────<  user_id             FK
signup_ts                                       session_id          PK
segment                                         session_date
country                                         variant             FK lógico
plan                                            traffic_source
channel                                         viewed_product      0/1
device                                          added_to_cart       0/1
                                                begin_checkout      0/1
                                                purchased           0/1
                                                revenue

experiments
────────────────────────
experiment_id
variant            clave lógica
variant_name
change_description
target_stage
start_date, end_date
```

| Tabla | Una fila representa | Uso |
|---|---|---|
| `users` | un usuario | segmentación |
| `events` | un evento | puente con el proyecto |
| `funnel_sessions` | una sesión experimental | funnel A/B |
| `experiments` | una variante | hipótesis y etapa objetivo |


## Preparación del entorno

Ejecuta estas celdas antes del breakout. No es necesario explicar la infraestructura línea por línea.


In [ ]:
# Infraestructura de la clase: ejecutar una vez.
%pip install -q duckdb pandas pyarrow requests


In [ ]:
from pathlib import Path
import requests
import pandas as pd
import duckdb

BASE_URL = "https://raw.githubusercontent.com/ljpiere/tpdata_python/refs/heads/main/DA/DA_students/datasets/"

DATASETS = {
    "users": "sprint04_users.parquet",
    "events": "sprint04_events.parquet",
    "funnel_sessions": "sprint04_funnel_sessions.parquet",
    "experiments": "sprint04_experiments.parquet",
}

def read_parquet_source(file_name: str) -> pd.DataFrame:
    """Lee primero el archivo local; si no existe, lo descarga temporalmente."""
    source_path = Path(file_name)

    if not source_path.exists():
        response = requests.get(f"{BASE_URL}{file_name}", timeout=30)
        response.raise_for_status()

        source_path = Path("/tmp") / file_name
        source_path.write_bytes(response.content)

    # DuckDB evita incompatibilidades de metadatos entre versiones de PyArrow.
    reader = duckdb.connect(database=":memory:")
    try:
        return reader.execute(
            "SELECT * FROM read_parquet(?)",
            [str(source_path)],
        ).df()
    finally:
        reader.close()

users = read_parquet_source(DATASETS["users"])
events = read_parquet_source(DATASETS["events"])
funnel_sessions = read_parquet_source(DATASETS["funnel_sessions"])
experiments = read_parquet_source(DATASETS["experiments"])

con = duckdb.connect(database=":memory:")
con.register("users_df", users)
con.register("events_df", events)
con.register("funnel_sessions_df", funnel_sessions)
con.register("experiments_df", experiments)

con.execute("CREATE OR REPLACE TABLE users AS SELECT * FROM users_df")
con.execute("CREATE OR REPLACE TABLE events AS SELECT * FROM events_df")
con.execute("CREATE OR REPLACE TABLE funnel_sessions AS SELECT * FROM funnel_sessions_df")
con.execute("CREATE OR REPLACE TABLE experiments AS SELECT * FROM experiments_df")

def sql(query: str) -> pd.DataFrame:
    return con.execute(query).df()

print("Tablas listas:", ", ".join(DATASETS.keys()))


In [ ]:
sql("""
SELECT 'users' AS table_name, COUNT(*) AS rows FROM users
UNION ALL
SELECT 'events', COUNT(*) FROM events
UNION ALL
SELECT 'funnel_sessions', COUNT(*) FROM funnel_sessions
UNION ALL
SELECT 'experiments', COUNT(*) FROM experiments;
""")


---
# Bloque 1 · Breakout rooms: explorar los cuatro datasets

**Tiempo:** 12 minutos.

Cada grupo debe completar este inventario:

| Tabla | ¿Qué representa una fila? | Llave | Dos columnas relevantes | Pregunta de negocio |
|---|---|---|---|---|
| `users` |  |  |  |  |
| `events` |  |  |  |  |
| `funnel_sessions` |  |  |  |  |
| `experiments` |  |  |  |  |

No realicen limpieza. Identifiquen estructura, categorías y relaciones.


## 1.1 Usuarios

**Pregunta:** ¿qué variables permitirían segmentar el desempeño del experimento?


In [ ]:
sql("""
SELECT *
FROM users
LIMIT 5;
""")


## 1.2 Eventos

**Pregunta:** ¿qué relación existe entre los eventos del proyecto y las etapas resumidas del experimento?


In [ ]:
sql("""
SELECT *
FROM events
ORDER BY user_id, event_ts
LIMIT 8;
""")


In [ ]:
sql("""
SELECT DISTINCT event_name
FROM events
ORDER BY event_name;
""")


## 1.3 Sesiones experimentales

**Pregunta:** ¿qué representa una fila y qué columnas codifican las etapas del funnel?


In [ ]:
sql("""
SELECT *
FROM funnel_sessions
ORDER BY session_date, session_id
LIMIT 8;
""")


## 1.4 Metadatos del experimento

**Pregunta:** ¿qué cambio distingue las variantes y cuál es la etapa objetivo?


In [ ]:
sql("""
SELECT *
FROM experiments
ORDER BY variant;
""")


## Preguntas del breakout

1. ¿Cuál es la granularidad de `funnel_sessions`?
2. ¿Qué columna la relaciona con `users`?
3. ¿Qué cambio distingue A de B?
4. ¿Qué etapa debe ser el foco?
5. ¿Los indicadores respetan el orden lógico?
6. ¿Qué diferencia hay entre sesiones y usuarios?

### Respuestas del grupo

1.  
2.  
3.  
4.  
5.  
6.


## Ejercicio 1.5 · Validar el orden lógico

Comprueba si existen sesiones que:

- añadieron al carrito sin ver producto;
- iniciaron checkout sin añadir al carrito;
- compraron sin iniciar checkout.


In [ ]:
# TODO — Completa el check lógico.
#
# Usa SUM(CASE WHEN condición THEN 1 ELSE 0 END)
# para crear tres indicadores:
#
# - cart_without_view
# - checkout_without_cart
# - purchase_without_checkout
#
# Tabla: funnel_sessions
#
# Escribe tu consulta debajo:


<details>
<summary><strong>Pistas del ejercicio 1.5</strong></summary>

- Una inconsistencia combina una etapa posterior igual a `1` con la anterior igual a `0`.
- Las tres métricas pueden calcularse en una sola consulta.
- El resultado esperado debe interpretarse antes de continuar.

</details>

### Hallazgos del breakout

- Granularidad de `users`: ____________________________________________
- Granularidad de `events`: ___________________________________________
- Granularidad de `funnel_sessions`: __________________________________
- Significado de B: ___________________________________________________
- Etapa objetivo: _____________________________________________________
- ¿Un usuario puede tener varias sesiones?: ____________________________


---
# Bloque 2 · Funnel A/B con el patrón de plataforma

```text
CTE viewed → CTE cart → CTE checkout → CTE purchase → LEFT JOIN → conteos
```

La unidad de análisis es la **sesión**.

## Ejercicio 2.1 · Conteos por etapa y variante

Antes de escribir:

- Tabla base: ______________________________
- Llave de unión: ___________________________
- Columna de agrupación: ____________________
- Primera etapa: ____________________________


In [ ]:
# TODO — Construye el funnel A/B.
#
# sql("""
# WITH viewed AS (
#     SELECT DISTINCT session_id, user_id, variant
#     FROM funnel_sessions
#     WHERE viewed_product = 1
# ),
# cart AS (
#     -- TODO
# ),
# checkout AS (
#     -- TODO
# ),
# purchase AS (
#     -- TODO: incluye revenue
# )
# SELECT
#     v.variant,
#     -- TODO: contar sesiones en cada etapa
#     -- TODO: sumar revenue
# FROM viewed AS v
# LEFT JOIN cart AS c ON ...
# LEFT JOIN checkout AS ch ON ...
# LEFT JOIN purchase AS p ON ...
# GROUP BY v.variant
# ORDER BY v.variant;
# """)
#
# Escribe tu solución debajo:


<details>
<summary><strong>Pistas del ejercicio 2.1</strong></summary>

1. Cada CTE filtra una columna binaria igual a `1`.
2. Usa `session_id` como llave del funnel.
3. Conserva `variant` para agrupar.
4. El revenue solo pertenece a sesiones con compra.
5. Usa `COALESCE(SUM(...), 0)` si necesitas evitar nulos.

</details>

### Validación de conteos

| Variante | Viewed | Cart | Checkout | Purchase | Revenue |
|---|---:|---:|---:|---:|---:|
| A |  |  |  |  |  |
| B |  |  |  |  |  |

- ¿Los conteos disminuyen al avanzar?: _________________________________
- ¿Qué variante tiene más sesiones?: __________________________________
- ¿Podemos decidir solo con revenue bruto?: ____________________________


## Ejercicio 2.2 · Tasas de conversión

Reutiliza el funnel anterior dentro de una CTE llamada `funnel_counts`.

Calcula:

- `view_to_cart_pct`;
- `cart_to_checkout_pct`;
- `checkout_to_purchase_pct`;
- `total_conversion_pct`.


In [ ]:
# TODO — Calcula las tasas.
#
# Usa esta lógica:
#
# etapa actual * 100.0 / NULLIF(etapa anterior, 0)
#
# La conversión total usa:
#
# sessions_purchase / sessions_viewed
#
# Escribe la consulta completa debajo:


### Lectura del funnel

Completa sin observar todavía la interpretación del docente.

| Métrica | Mejor variante | Diferencia relevante |
|---|---|---|
| View → Cart |  |  |
| Cart → Checkout |  |  |
| Checkout → Purchase |  |  |
| Conversión total |  |  |

```text
La variante ____ mejora las etapas tempranas, pero ______________________.
La etapa objetivo muestra ______________________________________________.
Por tanto, todavía no concluiría _______________________________________.
```


---
# Bloque 3 · Vincular la métrica con la hipótesis

## Ejercicio 3.1 · Evaluar la etapa objetivo

Construye un resumen con:

- sesiones que iniciaron checkout;
- sesiones que compraron;
- conversión `checkout → purchase`;
- revenue;
- nombre y descripción de la variante;
- etapa objetivo.


In [ ]:
# TODO — Une el resumen del funnel con `experiments`.
#
# Pasos:
# 1. Crea una CTE `funnel` agrupada por `variant`.
# 2. Calcula SUM(begin_checkout), SUM(purchased) y SUM(revenue).
# 3. Calcula purchased / begin_checkout.
# 4. Une `experiments` con `funnel` mediante `variant`.
#
# Escribe tu consulta debajo:


## Marco de decisión

| Componente | Mi respuesta |
|---|---|
| Decisión |  |
| Evidencia |  |
| Limitación |  |
| Siguiente paso |  |

### Pregunta de validación

¿Por qué es incorrecto afirmar “B es mejor porque genera más revenue”?

______________________________________________________________________

### Recomendación preliminar

```text
Recomendamos ________________________________.
La evidencia principal es _____________________________________________.
Todavía no podemos afirmar ____________________________________________.
Como siguiente paso proponemos ________________________________________.
```


---
# Bloque 4 · ¿El resultado cambia por país?

La segmentación sirve para generar hipótesis, no para declarar ganadores con muestras pequeñas.

## Ejercicio 4.1 · Funnel por país y variante

Une `funnel_sessions` con `users` y calcula:

- sesiones vistas;
- sesiones que iniciaron checkout;
- sesiones con compra;
- conversión `checkout → purchase`;
- conversión total;
- revenue.


In [ ]:
# TODO — Segmenta el resultado.
#
# Pasos:
# 1. Crea una CTE `base`.
# 2. Une por `user_id`.
# 3. Selecciona `country` y las columnas del funnel.
# 4. Agrupa por `country, variant`.
# 5. Calcula tasas con `NULLIF`.
#
# Escribe tu consulta debajo:


### Interpretación responsable

| País | Variante | Volumen | Checkout → Purchase | ¿Hipótesis para investigar? |
|---|---|---:|---:|---|
|  |  |  |  |  |
|  |  |  |  |  |
|  |  |  |  |  |
|  |  |  |  |  |

Antes de recomendar un rollout por país necesito revisar:  
______________________________________________________________________


## Correspondencia con la plataforma

| Plataforma | Práctica |
|---|---|
| explorar esquema y flujo | breakout de cuatro tablas |
| identificar etapas | columnas 0/1 |
| una CTE por etapa | `viewed`, `cart`, `checkout`, `purchase` |
| unir etapas | `LEFT JOIN` |
| conversiones | `funnel_counts` y `NULLIF` |
| segmentación | `GROUP BY country, variant` |
| interpretación | decisión, evidencia, limitación, siguiente paso |


## Errores frecuentes

| Señal | Causa | Acción |
|---|---|---|
| B tiene más compras y se declara ganador | se compararon totales | calcular tasas |
| Checkout usa vistas como denominador | denominador incorrecto | usar checkouts |
| Conteos crecen al avanzar | llave incorrecta | unir por `session_id` |
| Revenue contradice conversión | mide volumen y ticket | separar métricas |
| 100 % con pocas sesiones | muestra inestable | mostrar volumen |
| Se afirma causalidad | falta inferencia | presentar como descriptivo |


---
# Cierre · Validación de conocimiento

1. ¿Qué problema motivó el experimento?
2. ¿Qué representa una fila de `funnel_sessions`?
3. ¿Por qué usamos una CTE por etapa?
4. ¿Cuál es el denominador de `checkout_to_purchase_pct`?
5. ¿Qué hallazgo contradice la intención de B?
6. ¿Por qué no basta el revenue?
7. ¿Qué recomendación es prudente?

### Mis respuestas

1.  
2.  
3.  
4.  
5.  
6.  
7.  

**Tema que necesito repasar:** _________________________________________


## Takeaways

Completa con tus palabras:

- El contexto de negocio debe aparecer antes de ________________________.
- Explorar un dataset significa _______________________________________.
- Una CTE por etapa permite ___________________________________________.
- La métrica principal debe ___________________________________________.
- Revenue y conversión se diferencian porque __________________________.
- Una recomendación responsable incluye _______________________________.


### ¿Necesitas ayuda?

- `OFFICE HOURS`: revisión de consultas.
- `DA_CONSULTA`: dudas sobre proyecto.
- Comparte pregunta, consulta, resultado y expectativa.
